# CAARMA Gender-Aware W&B Training

This notebook downloads the VoxCeleb data, creates a gender-aware CAARMA config, logs into Weights & Biases, and trains with `mixup_gender.py` using `sl_mixup`.

In [ ]:
%pip install -q kaggle wandb pytorch-lightning==2.5.0.post0 librosa==0.10.0 soundfile==0.12.1 scikit-learn PyYAML pandas speechbrain==1.0.3

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import urllib.request

import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "mixup_gender.py").exists(), "Run this notebook from the CAARMA repo root."

DATA_DIR = Path("/content/data") if Path("/content").exists() else PROJECT_ROOT / "data"
TRAIN_ROOT = DATA_DIR / "voxceleb1" / "vox1_dev_wav" / "wav"
TEST_DIR = Path("/content/voxceleb1_test") if Path("/content").exists() else PROJECT_ROOT / "voxceleb1_test"
TEST_ROOT = TEST_DIR / "wav"

TRIAL_PATH = PROJECT_ROOT / "data" / "vox1_test.txt"
TRAIN_CSV = Path("/content/voxceleb_full.csv") if Path("/content").exists() else PROJECT_ROOT / "voxceleb_full.csv"
META_PATH = PROJECT_ROOT / "vox1_meta.csv"
CONFIG_PATH = PROJECT_ROOT / "config_gender.yaml"
SAVE_DIR = PROJECT_ROOT / "caarma_gender_ckpts"

USE_WANDB = True
WANDB_PROJECT = "mixup"
WANDB_RUN_NAME = "caarma_gender"

def run(command):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    subprocess.run(command, check=True)


Upload `kaggle.json` when prompted. The notebook will copy it into `~/.kaggle/kaggle.json` with the correct permissions.

In [ ]:
try:
    from google.colab import files
except ImportError:
    files = None

local_kaggle = PROJECT_ROOT / "kaggle.json"
home_kaggle = Path.home() / ".kaggle" / "kaggle.json"

if not local_kaggle.exists() and not home_kaggle.exists() and files is not None:
    files.upload()

home_kaggle.parent.mkdir(parents=True, exist_ok=True)
if local_kaggle.exists():
    shutil.copy(local_kaggle, home_kaggle)
    home_kaggle.chmod(0o600)

if not home_kaggle.exists():
    raise FileNotFoundError("Missing Kaggle credentials. Upload kaggle.json or place it at ~/.kaggle/kaggle.json.")

print("Kaggle credentials ready:", home_kaggle)

Run this W&B login cell before training. `mixup_gender.py` will create a `WandbLogger` because the config written below sets `USE_WANDB: True`.

In [ ]:
import wandb

if USE_WANDB:
    wandb.login()
    print("W&B login complete")

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
archive = DATA_DIR / "voxceleb1.zip"

if not TRAIN_ROOT.exists():
    if not archive.exists():
        run(["kaggle", "datasets", "download", "-d", "namhocayai/voxceleb1", "-p", DATA_DIR])
    run(["unzip", "-q", "-n", archive, "-d", DATA_DIR / "voxceleb1"])

print("Training wav root:", TRAIN_ROOT)
assert TRAIN_ROOT.exists(), f"Expected training wav root not found: {TRAIN_ROOT}"

In [ ]:
TRIAL_PATH.parent.mkdir(parents=True, exist_ok=True)
if not TRIAL_PATH.exists():
    urllib.request.urlretrieve(
        "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test.txt",
        TRIAL_PATH,
    )

TEST_DIR.mkdir(parents=True, exist_ok=True)
if not TEST_ROOT.exists():
    test_archives = list(TEST_DIR.glob("*.zip"))
    if not test_archives:
        run(["kaggle", "datasets", "download", "-d", "kryakrya/voxceleb1test", "-p", TEST_DIR])
        test_archives = list(TEST_DIR.glob("*.zip"))
    for test_archive in test_archives:
        run(["unzip", "-q", "-n", test_archive, "-d", TEST_DIR])

print("Trial file:", TRIAL_PATH)
print("Test wav root:", TEST_ROOT)
assert TRIAL_PATH.exists(), f"Trial file not found: {TRIAL_PATH}"
assert TEST_ROOT.exists(), f"Expected test wav root not found: {TEST_ROOT}"

In [ ]:
if not META_PATH.exists():
    urllib.request.urlretrieve(
        "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/vox1_meta.csv",
        META_PATH,
    )

meta = pd.read_csv(META_PATH, sep=None, engine="python")
print("vox1_meta.csv:", META_PATH)
meta.head()

In [ ]:
speakers = sorted(speaker.name for speaker in TRAIN_ROOT.iterdir() if speaker.is_dir())

rows = []
for label, speaker in enumerate(speakers):
    for wav_path in sorted((TRAIN_ROOT / speaker).rglob("*.wav")):
        rows.append({
            "utt_spk_int_labels": label,
            "utt_spk_id": speaker,
            "utt_paths": str(wav_path),
        })

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError(f"No wav files found under {TRAIN_ROOT}")

df.to_csv(TRAIN_CSV, index=False)
num_spk = int(df["utt_spk_int_labels"].nunique())

print("Wrote:", TRAIN_CSV)
print("Speakers:", num_spk)
print("Utterances:", len(df))
df.head()

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "model": "MFA-CONFORMER",
    "features": "Fbank",
    "augmentations": {
        "add_noise": False,
        "add_reverb": False,
        "drop_freq": False,
        "drop_chunk": False,
    },
    "dataset": str(TRAIN_CSV),
    "USE_WANDB": USE_WANDB,
    "wandb_project": WANDB_PROJECT,
    "criterion": "AMSoftmaxGAN",
    "init_lr": 0.001,
    "epochs": 30,
    "weight_decay": 0.0000001,
    "trial_path": str(TRIAL_PATH),
    "root": str(TEST_ROOT) + os.sep,
    "warmup_step": 2000,
    "batch_size": 50,
    "num_workers": 4,
    "second": 3,
    "do_augmentation": False,
    "num_spk": num_spk,
    "mixup": True,
    "sl_mixup": True,
    "vox1_meta_path": str(META_PATH),
    "embedding_dim": 192,
    "checkpoint_path": "None",
    "save_dir": str(SAVE_DIR),
    "title": WANDB_RUN_NAME,
    "score_output_prefix": "caarma_gender",
}

CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH)
print(CONFIG_PATH.read_text())

In [ ]:
trials = np.loadtxt(TRIAL_PATH, str)
example_eval_wav = TEST_ROOT / trials[0][1]

print("First trial:", trials[0])
print("Example eval wav:", example_eval_wav)
print("Exists:", example_eval_wav.exists())
assert example_eval_wav.exists(), "Test data root does not match the trial file paths."

pd.read_csv(TRAIN_CSV, nrows=5)

In [ ]:
!python mixup_gender.py --config config_gender.yaml --mode train --sl-mixup

In [ ]:
# Optional: test after training with a checkpoint.
# CKPT_PATH = "caarma_gender_ckpts/path_to_checkpoint.ckpt"
# !python mixup_gender.py --config config_gender.yaml --mode test --checkpoint-path {CKPT_PATH} --sl-mixup